# TP 3 — Détective réseau : du symptôme à l'hypothèse réfutable

**Public cible :** étudiant, technicien ou NetOps débutant connaissant débit,
délai, perte et taille de paquet. **Durée : 60 minutes.** Aucun accès réseau,
privilège, fichier ni outil externe n'est utilisé.

> Mission : recevoir quatre tickets ambigus, prédire les signatures, choisir des
> mesures discriminantes, falsifier les causes incompatibles et produire un
> rapport d'incident défendable. Le but n'est pas de deviner une étiquette.


## Contrat d'apprentissage

À la fin du TP, vous saurez **prédire → mesurer → falsifier → diagnostiquer →
transférer**. Vous distinguerez quatre signatures synthétiques :

1. **capacité** : le goodput plafonne quand la charge augmente ;
2. **bufferbloat** : le RTT sous charge explose alors que le goodput reste élevé ;
3. **pertes radio-like** : les pertes arrivent par épisodes, sans seuil de taille net ;
4. **trou noir MTU** : les petits paquets passent et les grands paquets avec
   **DF** échouent au-delà d'un seuil.

Avant toute cellule de révélation, écrivez une prédiction et le test qui pourrait
la contredire. Règle : jamais un verdict sur un seul KPI. Chaque conclusion doit
proposer une **contre-expérience** et une prochaine mesure indépendante.


In [ ]:
import base64
import hashlib
import html
import random
import statistics

SEED = 73

PROFILES = {
    "capacity": {
        "kind": "capacity", "capacity_mbps": 6.0, "base_rtt_ms": 18.0,
        "mtu_payload": 9000, "radio_loss": 0.0,
    },
    "bufferbloat": {
        "kind": "bufferbloat", "capacity_mbps": 12.0, "base_rtt_ms": 18.0,
        "mtu_payload": 9000, "radio_loss": 0.0,
    },
    "radio_like": {
        "kind": "radio_like", "capacity_mbps": 20.0, "base_rtt_ms": 28.0,
        "mtu_payload": 9000, "radio_loss": 0.08,
    },
    "mtu_blackhole": {
        "kind": "mtu_blackhole", "capacity_mbps": 20.0, "base_rtt_ms": 20.0,
        "mtu_payload": 1360, "radio_loss": 0.0,
    },
}

profile_names = list(PROFILES)
random.Random(SEED).shuffle(profile_names)
INCIDENTS = dict(zip("ABCD", profile_names))

# Cas X : mêmes mécanismes, valeurs inédites. Son étiquette n'est révélée qu'après le test.
TRANSFER_PROFILE = {
    "kind": "radio_like", "capacity_mbps": 16.0, "base_rtt_ms": 34.0,
    "mtu_payload": 9000, "radio_loss": 0.11,
}

class RichHTML:
    def __init__(self, markup):
        self.markup = markup

    def _repr_html_(self):
        return self.markup

print("Tickets reçus :", ", ".join(INCIDENTS), "+ transfert X")
print("Seed pédagogique :", SEED, "— causes masquées jusqu'au verdict.")


## 1. Prédire avant de mesurer

Complétez mentalement ou dans une copie du notebook :

| Hypothèse | Prédiction qui la soutient | Observation qui la falsifie |
|---|---|---|
| capacité | goodput plafonné sous charge | seuil brutal dépendant de la taille |
| bufferbloat | RTT au repos bas, RTT sous charge très haut | RTT stable sous forte charge |
| pertes radio-like | pertes temporelles corrélées sur plusieurs tailles | perte uniquement au-dessus d'une taille précise |
| trou noir MTU | petits DF livrés, grands DF perdus | toutes les tailles ont un comportement progressif |

**Choix des sondes :** balayage de charge ; RTT simultané ; répétitions dans le
temps ; balayage de taille avec DF. Chacune ne change qu'une variable.


In [ ]:
LOADS_MBPS = (1.0, 3.0, 6.0, 9.0, 12.0, 18.0)
PAYLOAD_SIZES = (256, 800, 1200, 1360, 1400, 1472)
REPEAT_SAMPLES = tuple(range(12))

def stable_unit(*parts):
    payload = "|".join(str(part) for part in (SEED,) + parts).encode("utf-8")
    integer = int.from_bytes(hashlib.sha256(payload).digest()[:8], "big")
    return integer / float(2**64 - 1)

def probe(profile, label, offered_mbps, payload_bytes, sample=0, df=True, corrected=False):
    capacity = profile["capacity_mbps"]
    ratio = offered_mbps / capacity
    noise = stable_unit(label, offered_mbps, payload_bytes, sample, df, corrected) - 0.5
    kind = profile["kind"]

    if kind == "mtu_blackhole" and df and payload_bytes > profile["mtu_payload"]:
        pdr = 0.0
    elif kind == "radio_like":
        burst = 0.16 if sample % 8 in (2, 3, 4) else 0.035
        loss = profile["radio_loss"] + burst + 0.02 * noise
        if corrected:
            loss *= 0.2
        pdr = 100.0 * (1.0 - min(0.45, max(0.0, loss)))
    elif kind == "capacity" and ratio > 1.0:
        pdr = 100.0 / ratio
    else:
        pdr = 99.4 + 0.4 * noise

    serialization_ms = payload_bytes * 8.0 / (capacity * 1000.0)
    overload = max(0.0, ratio - 0.65)
    if kind == "bufferbloat":
        queue_ms = min(260.0, 190.0 * overload * overload)
        if corrected:
            queue_ms = min(queue_ms, 18.0)
    elif kind == "capacity":
        queue_ms = min(35.0, 18.0 * max(0.0, ratio - 1.0))
    else:
        queue_ms = 2.0 * max(0.0, ratio - 0.8)
    rtt_ms = 0.0 if pdr == 0.0 else profile["base_rtt_ms"] + serialization_ms + queue_ms + noise
    goodput = min(capacity * 0.995, offered_mbps * pdr / 100.0)
    return {
        "offered_mbps": offered_mbps,
        "payload_bytes": payload_bytes,
        "sample": sample,
        "df": df,
        "goodput_mbps": round(max(0.0, goodput), 3),
        "rtt_ms": round(max(0.0, rtt_ms), 2),
        "pdr_pct": round(min(100.0, max(0.0, pdr)), 2),
    }

assert probe(PROFILES["mtu_blackhole"], "check", 3.0, 1472)["pdr_pct"] == 0.0
assert probe(PROFILES["mtu_blackhole"], "check", 3.0, 1200)["pdr_pct"] > 99.0


## 2. Mesurer avec quatre sondes mono-variable

Le simulateur ci-dessous n'est pas un réseau réel. Il fabrique des observations
cohérentes afin d'apprendre à choisir une mesure. Pour chaque ticket :

- la charge varie à taille constante ;
- le RTT est observé au repos et sous charge ;
- douze répétitions cherchent une structure temporelle des pertes ;
- la taille varie avec DF à charge constante.

Notez d'abord ce qui est observé, sans nommer la cause.


In [ ]:
def run_suite(profile, label):
    return {
        "load": [probe(profile, label, load, 1000, sample=0) for load in LOADS_MBPS],
        "size": [probe(profile, label, 3.0, size, sample=1, df=True) for size in PAYLOAD_SIZES],
        "repeat": [probe(profile, label, 3.0, 1000, sample=sample) for sample in REPEAT_SAMPLES],
    }

SUITES = {
    case_id: run_suite(PROFILES[profile_name], case_id)
    for case_id, profile_name in INCIDENTS.items()
}
TRANSFER_SUITE = run_suite(TRANSFER_PROFILE, "X")

for case_id, suite in SUITES.items():
    load = suite["load"]
    size = suite["size"]
    print(
        f"Ticket {case_id}: goodput max={max(row['goodput_mbps'] for row in load):.2f} Mb/s; "
        f"RTT repos/charge={load[0]['rtt_ms']:.1f}/{load[-1]['rtt_ms']:.1f} ms; "
        f"PDR tailles={min(row['pdr_pct'] for row in size):.1f}–{max(row['pdr_pct'] for row in size):.1f}%"
    )


In [ ]:
def extract_features(suite):
    load = suite["load"]
    size = suite["size"]
    repeat = suite["repeat"]
    cliffs = [row["payload_bytes"] for row in size if row["pdr_pct"] < 10.0]
    repeat_losses = [100.0 - row["pdr_pct"] for row in repeat]
    return {
        "max_goodput_mbps": round(max(row["goodput_mbps"] for row in load), 2),
        "rtt_growth_ms": round(load[-1]["rtt_ms"] - load[0]["rtt_ms"], 1),
        "high_load_pdr_pct": load[-1]["pdr_pct"],
        "repeat_loss_mean_pct": round(statistics.mean(repeat_losses), 1),
        "repeat_loss_span_pct": round(max(repeat_losses) - min(repeat_losses), 1),
        "size_cliff_bytes": min(cliffs) if cliffs else None,
        "small_packet_pdr_pct": size[0]["pdr_pct"],
    }

def diagnose(features):
    if features["size_cliff_bytes"] is not None:
        return "mtu_blackhole"
    if features["rtt_growth_ms"] > 100.0 and features["high_load_pdr_pct"] > 90.0:
        return "bufferbloat"
    if features["repeat_loss_mean_pct"] > 8.0 and features["repeat_loss_span_pct"] > 8.0:
        return "radio_like"
    return "capacity"

FEATURES = {case_id: extract_features(suite) for case_id, suite in SUITES.items()}
VERDICTS = {case_id: diagnose(features) for case_id, features in FEATURES.items()}
TRANSFER_FEATURES = extract_features(TRANSFER_SUITE)
TRANSFER_VERDICT = diagnose(TRANSFER_FEATURES)

assert {VERDICTS[case_id] for case_id in VERDICTS} == set(PROFILES)
assert all(VERDICTS[case_id] == PROFILES[INCIDENTS[case_id]]["kind"] for case_id in INCIDENTS)
assert TRANSFER_VERDICT == TRANSFER_PROFILE["kind"]

for case_id in "ABCD":
    print(f"Ticket {case_id}: verdict soutenu = {VERDICTS[case_id]}")


## 3. Falsifier, puis diagnostiquer

Un score n'est pas une certitude. Pour chaque ticket, formulez :

- l'observation qui soutient le verdict ;
- au moins deux hypothèses éliminées et pourquoi ;
- le **falsificateur** qui ferait abandonner votre verdict ;
- la prochaine mesure indépendante sur un banc autorisé.

Le tableau de bord révèle les causes injectées seulement après les assertions.
Il rend le raisonnement visible ; il ne transforme pas quatre règles en IA.


In [ ]:
COLORS = {
    "capacity": "#0ea5e9",
    "bufferbloat": "#f97316",
    "radio_like": "#a855f7",
    "mtu_blackhole": "#ef4444",
}
LABELS = {
    "capacity": "Capacité",
    "bufferbloat": "Bufferbloat",
    "radio_like": "Pertes radio-like",
    "mtu_blackhole": "Trou noir MTU",
}

def points(rows, x0, y0, width, height, x_key, y_key, y_max):
    values = []
    for index, row in enumerate(rows):
        x = x0 + width * index / max(1, len(rows) - 1)
        y = y0 + height * (1.0 - min(y_max, max(0.0, row[y_key])) / y_max)
        values.append(f"{x:.1f},{y:.1f}")
    return " ".join(values)

def dashboard_svg():
    parts = [
        '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1200 790" role="img" '
        'aria-labelledby="detective-title detective-desc">',
        '<title id="detective-title">Tableau de bord Détective réseau</title>',
        '<desc id="detective-desc">Quatre tickets, mesures discriminantes et verdicts synthétiques.</desc>',
        '<rect width="1200" height="790" rx="28" fill="#07111f"/>',
        '<text x="55" y="64" fill="#f8fafc" font-size="32" font-family="sans-serif" font-weight="700">'
        'Détective réseau — mesurer avant de conclure</text>',
        '<text x="55" y="96" fill="#94a3b8" font-size="16" font-family="sans-serif">'
        'Charge → RTT · répétitions → pertes · taille + DF → seuil · verdict → contre-expérience</text>',
    ]
    for index, case_id in enumerate("ABCD"):
        column, row_index = index % 2, index // 2
        x0, y0 = 45 + column * 575, 130 + row_index * 300
        suite, feature, verdict = SUITES[case_id], FEATURES[case_id], VERDICTS[case_id]
        color = COLORS[verdict]
        cliff = feature["size_cliff_bytes"]
        parts.extend([
            f'<rect x="{x0}" y="{y0}" width="535" height="265" rx="20" fill="#101f33" stroke="{color}" stroke-width="2"/>',
            f'<rect x="{x0}" y="{y0}" width="535" height="12" rx="6" fill="{color}"/>',
            f'<text x="{x0 + 24}" y="{y0 + 48}" fill="#f8fafc" font-size="24" font-family="sans-serif" font-weight="700">Ticket {case_id}</text>',
            f'<text x="{x0 + 180}" y="{y0 + 48}" fill="{color}" font-size="20" font-family="sans-serif" font-weight="700">{html.escape(LABELS[verdict])}</text>',
            f'<text x="{x0 + 24}" y="{y0 + 82}" fill="#cbd5e1" font-size="15" font-family="monospace">goodput max {feature["max_goodput_mbps"]:.1f} Mb/s</text>',
            f'<text x="{x0 + 24}" y="{y0 + 108}" fill="#cbd5e1" font-size="15" font-family="monospace">croissance RTT {feature["rtt_growth_ms"]:.1f} ms</text>',
            f'<text x="{x0 + 24}" y="{y0 + 134}" fill="#cbd5e1" font-size="15" font-family="monospace">perte répétée {feature["repeat_loss_mean_pct"]:.1f}%</text>',
            f'<text x="{x0 + 24}" y="{y0 + 160}" fill="#cbd5e1" font-size="15" font-family="monospace">seuil taille {cliff if cliff else "aucun"}</text>',
            f'<text x="{x0 + 286}" y="{y0 + 82}" fill="#94a3b8" font-size="13" font-family="sans-serif">RTT selon la charge</text>',
            f'<polyline points="{points(suite["load"], x0 + 288, y0 + 98, 215, 58, "offered_mbps", "rtt_ms", 300.0)}" fill="none" stroke="{color}" stroke-width="4"/>',
            f'<line x1="{x0 + 288}" y1="{y0 + 158}" x2="{x0 + 503}" y2="{y0 + 158}" stroke="#334155"/>',
            f'<text x="{x0 + 286}" y="{y0 + 190}" fill="#94a3b8" font-size="13" font-family="sans-serif">PDR selon taille + DF</text>',
            f'<polyline points="{points(suite["size"], x0 + 288, y0 + 202, 215, 42, "payload_bytes", "pdr_pct", 100.0)}" fill="none" stroke="#22c55e" stroke-width="4"/>',
            f'<line x1="{x0 + 288}" y1="{y0 + 246}" x2="{x0 + 503}" y2="{y0 + 246}" stroke="#334155"/>',
            f'<circle cx="{x0 + 24}" cy="{y0 + 218}" r="7" fill="{color}"/>',
            f'<text x="{x0 + 42}" y="{y0 + 224}" fill="#e2e8f0" font-size="14" font-family="sans-serif">cause soutenue, pas prouvée terrain</text>',
        ])
    parts.extend([
        '<rect x="45" y="735" width="1110" height="34" rx="12" fill="#15283e"/>',
        '<text x="600" y="758" text-anchor="middle" fill="#e2e8f0" font-size="15" font-family="sans-serif">'
        'Une cause reste acceptable seulement tant que ses prédictions résistent aux tests discriminants.</text>',
        '</svg>',
    ])
    return "".join(parts)

DASHBOARD = dashboard_svg()
assert DASHBOARD.count("Ticket ") == 4 and "<svg" in DASHBOARD
RichHTML(DASHBOARD)


## 4. Transférer et chercher la contre-expérience

Le ticket X change capacité, RTT de base et niveau de perte. Réutilisez la
méthode, pas les valeurs mémorisées. Ensuite, chaque diagnostic reçoit une
contre-expérience corrective qui ne change qu'un mécanisme : charge réduite,
file bornée, pertes atténuées ou taille adaptée au chemin.

Un résultat qui ne change pas comme prévu doit faire rouvrir le diagnostic.


In [ ]:
def counter_experiment(profile, label):
    kind = profile["kind"]
    if kind == "capacity":
        before = probe(profile, label, 18.0, 1000)
        after = probe(profile, label, 5.0, 1000)
        metric = "PDR sous charge"
    elif kind == "bufferbloat":
        before = probe(profile, label, 18.0, 1000)
        after = probe(profile, label, 18.0, 1000, corrected=True)
        metric = "RTT sous charge"
    elif kind == "radio_like":
        before = probe(profile, label, 3.0, 1000, sample=3)
        after = probe(profile, label, 3.0, 1000, sample=3, corrected=True)
        metric = "PDR pendant épisode"
    else:
        before = probe(profile, label, 3.0, 1472, sample=1, df=True)
        after = probe(profile, label, 3.0, 1200, sample=1, df=True)
        metric = "PDR taille testée"
    return metric, before, after

def rejected_hypotheses(verdict):
    return [LABELS[name] for name in LABELS if name != verdict]

def report_for(case_id, profile, suite):
    features = extract_features(suite)
    verdict = diagnose(features)
    metric, before, after = counter_experiment(profile, case_id)
    cliff = features["size_cliff_bytes"] or "aucun"
    return f'''## Ticket {case_id}

- Observation : goodput max {features['max_goodput_mbps']:.1f} Mb/s ; croissance RTT {features['rtt_growth_ms']:.1f} ms ; perte répétée {features['repeat_loss_mean_pct']:.1f} % ; seuil taille {cliff}.
- Hypothèse soutenue : {LABELS[verdict]}.
- Hypothèses rejetées : {', '.join(rejected_hypotheses(verdict))}.
- Contre-expérience : {metric} ; avant PDR={before['pdr_pct']:.1f} %, RTT={before['rtt_ms']:.1f} ms ; après PDR={after['pdr_pct']:.1f} %, RTT={after['rtt_ms']:.1f} ms.
- Falsificateur : absence de l'effet prédit après répétitions indépendantes.
- Prochaine mesure : reproduire sur un banc autorisé avec timestamps, captures et configuration versionnée.
- Frontière : cause injectée synthétique ; aucun diagnostic de production.
'''

reports = [
    report_for(case_id, PROFILES[INCIDENTS[case_id]], SUITES[case_id])
    for case_id in "ABCD"
]
reports.append(report_for("X", TRANSFER_PROFILE, TRANSFER_SUITE))
incident_report = "# Rapport d'incident — TP Détective réseau\n\n" + "\n---\n\n".join(reports)
report_digest = hashlib.sha256(incident_report.encode("utf-8")).hexdigest()
encoded_report = base64.b64encode(incident_report.encode("utf-8")).decode("ascii")

rows = []
for case_id in "ABCD":
    metric, before, after = counter_experiment(PROFILES[INCIDENTS[case_id]], case_id)
    rows.append(
        f"<tr><td>{case_id}</td><td>{html.escape(metric)}</td>"
        f"<td>{before['pdr_pct']:.1f}% / {before['rtt_ms']:.1f} ms</td>"
        f"<td>{after['pdr_pct']:.1f}% / {after['rtt_ms']:.1f} ms</td></tr>"
    )

report_html = f'''
<div style="font-family:system-ui;padding:18px;border:1px solid #38bdf8;border-radius:14px;background:#effaff">
  <h3 style="margin-top:0">Transfert X : {html.escape(LABELS[TRANSFER_VERDICT])}</h3>
  <p>Le mécanisme est retrouvé avec des valeurs inédites. Empreinte du rapport : <code>{report_digest[:16]}</code>.</p>
  <table style="border-collapse:collapse;width:100%"><thead><tr><th>Ticket</th><th>Contre-expérience</th><th>Avant</th><th>Après</th></tr></thead>
  <tbody>{''.join(rows)}</tbody></table>
  <p><a download="rapport_incident_detective_reseau.md" href="data:text/markdown;base64,{encoded_report}">⬇ Télécharger le rapport d'incident</a></p>
</div>
'''
assert len(incident_report) > 2500 and "Ticket X" in incident_report
RichHTML(report_html)


## 5. Auto-évaluation et limites de preuve

Barème conseillé sur 10 : prédictions écrites (2), sondes mono-variable (2),
falsification de causes concurrentes (2), verdict et incertitude (2), transfert
et contre-expérience (2). Une bonne étiquette sans justification vaut au plus 2.

### Limites

- données et défauts entièrement **synthétiques** ; aucune capture terrain ;
- modèle de goodput et de file simplifié, sans TCP, noyau ni application réelle ;
- « pertes radio-like » décrit une signature injectée, pas une preuve RF : aucun
  PHY/MAC, RSSI, SNR, interférent ou mouvement ;
- le seuil MTU imite un symptôme PMTUD/DF, sans ICMP Packet Too Big ni fragmentation réelle ;
- les règles ne considèrent que quatre hypothèses et ne sont ni une IA, ni un
  diagnostic opérationnel ; goodput et capacité physique ne sont pas synonymes ;
- le rapport d'incident est un artefact pédagogique, jamais une autorisation d'action.

Pour une preuve réelle, répéter les mesures sur un banc autorisé, documenter les
versions et confronter le modèle à des captures indépendantes.
